# Test Guardrail: OFF-TOPIC Detection

Test các trường hợp off-topic để kiểm tra guardrail hoạt động đúng.

**Test Cases:**
- Off-topic: thời tiết, thể thao, giải trí
- Off-topic: câu hỏi cá nhân về bot
- Off-topic: chính trị, tin tức
- On-topic: hướng nghiệp (để so sánh)
- Edge cases: có từ khóa off-topic nhưng context hướng nghiệp

## 1. Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project root:", ROOT)

Project root: C:\Users\ADMIN\OneDrive\Documents\PtitCareer\chatbot-service


In [2]:
from dotenv import load_dotenv

load_dotenv(ROOT / ".env")
print("Loaded .env")

Loaded .env


In [3]:
from app.services.chat_service import get_chat_service
from app.core.memory.long_term import load_profile_from_file, load_holland_from_file

# Khởi tạo ChatService
chat = get_chat_service()
print("ChatService initialized")

ChatService initialized


## 2. Test Cases

In [5]:
# Test cases: OFF-TOPIC
test_cases_off_topic = [
    # Thời tiết, thể thao
    "Hôm nay trời đẹp quá",
    "Bạn có xem bóng đá không?",
    "World Cup năm nay đội nào thắng?",
    
    # Giải trí
    "Phim mới hay không?",
    "Bài hát này tên gì?",
    "Netflix có phim gì hay?",
    
    # Câu hỏi về bot
    "Bạn tên gì?",
    "Bạn bao nhiêu tuổi?",
    "Bạn ở đâu?",
    
    # Chính trị, tin tức
    "Tin tức hôm nay có gì?",
    "Bầu cử năm nay như thế nào?",
    
    # Tán gẫu
    "Ờ",
    "Ok",
    "Được rồi"
]

# Test cases: ON-TOPIC (để so sánh)
test_cases_on_topic = [
    "Tư vấn cho tôi ngành công nghệ thông tin",
    "Điểm chuẩn ĐH Bách Khoa năm 2025?",
    "Tôi muốn học ngành marketing số",
    "Ngành y học những gì?",
    "Holland test là gì?",
    "Tôi có kết quả Holland RIASEC, gợi ý nghề cho tôi"
]

# Edge cases: có từ khóa off-topic nhưng context hướng nghiệp
test_cases_edge = [
    "Tôi muốn học ngành âm nhạc",  # có "âm nhạc" nhưng context hướng nghiệp
    "Ngành phim ảnh học gì?",  # có "phim ảnh" nhưng context hướng nghiệp
    "Xin chào, tư vấn cho tôi ngành IT"  # có chào hỏi nhưng có câu hỏi hướng nghiệp
]

print(f"Test cases OFF-TOPIC: {len(test_cases_off_topic)}")
print(f"Test cases ON-TOPIC: {len(test_cases_on_topic)}")
print(f"Test cases EDGE: {len(test_cases_edge)}")

Test cases OFF-TOPIC: 14
Test cases ON-TOPIC: 6
Test cases EDGE: 3


## 3. Test OFF-TOPIC Cases

In [6]:
import uuid

def test_message(message: str, thread_id: str = None, profile_data=None, holland_data=None):
    """Test message và check guardrail."""
    if thread_id is None:
        thread_id = str(uuid.uuid4())
    
    print(f"\n{'='*60}")
    print(f"Input: {message}")
    print(f"{'='*60}")
    
    try:
        response = chat.chat(message=message, thread_id=thread_id, profile_data=profile_data, holland_data=holland_data)
        print(f"\n{response}\n")
        
        if "lệch scope hướng nghiệp" in response.lower():
            print("GUARDRAIL REDIRECT")
        else:
            print("No redirect")
    except Exception as e:
        print(f"Error: {e}")

In [8]:
# Test từng off-topic case
for i, message in enumerate(test_cases_off_topic[:3], 1):  # Test 3 cases đầu tiên
    print(f"\n\n### Test {i}: OFF-TOPIC")
    test_message(message)



### Test 1: OFF-TOPIC

Input: Hôm nay trời đẹp quá

Case này hơi lệch scope hướng nghiệp của mình rồi  
Mình đang tập trung giúp bạn chọn ngành/chọn nghề cho đúng.  
Bạn muốn đi theo hướng nào nè:  
1) Bạn đang thích ngành gì và đang phân vân điều gì?  
2) Bạn mạnh môn nào / thích hoạt động nào nhất?  
3) Bạn có kết quả Holland (RIASEC) chưa để mình gợi ý nghề?

GUARDRAIL REDIRECT


### Test 2: OFF-TOPIC

Input: Bạn có xem bóng đá không?

Case này hơi lệch scope hướng nghiệp của mình rồi  
Mình đang tập trung giúp bạn chọn ngành/chọn nghề cho đúng.  
Bạn muốn đi theo hướng nào nè:  
1) Bạn đang thích ngành gì và đang phân vân điều gì?  
2) Bạn mạnh môn nào / thích hoạt động nào nhất?  
3) Bạn có kết quả Holland (RIASEC) chưa để mình gợi ý nghề?

GUARDRAIL REDIRECT


### Test 3: OFF-TOPIC

Input: World Cup năm nay đội nào thắng?

Case này hơi lệch scope hướng nghiệp của mình rồi   
Mình đang tập trung giúp bạn chọn ngành/chọn nghề cho đúng.  
Bạn muốn đi theo hướng nào nè:
1) Bạn đang

## 4. Test ON-TOPIC Cases (So sánh)

In [10]:
# Test on-topic cases (không nên có guardrail redirect)
for i, message in enumerate(test_cases_on_topic[:2], 1):  # Test 2 cases đầu tiên
    print(f"\n\n### Test {i}: ON-TOPIC")
    test_message(message)



### Test 1: ON-TOPIC

Input: Tư vấn cho tôi ngành công nghệ thông tin

Ok, bạn có kết quả Holland (RIASEC) rồi đúng không? Mình sẽ dựa vào điểm số của bạn để tư vấn ngành Công nghệ Thông tin cho chuẩn nhé.

Nhưng trước hết, ngành Công nghệ Thông tin thường phù hợp với nhóm tính cách Investigative (I) – thích phân tích, giải quyết vấn đề, và Realistic (R) – thích làm việc thực tế, với máy móc, công cụ. Nếu bạn có điểm I và R cao, thì đây là ngành rất hợp với bạn.

Cụ thể, ngành này đòi hỏi bạn:
- Tư duy logic, khả năng phân tích tốt
- Thích khám phá, học hỏi công nghệ mới
- Có thể làm việc độc lập hoặc nhóm
- Chịu được áp lực deadline, liên tục cập nhật kiến thức

Bạn có điểm R, I, hay nhóm nào nổi bật trong kết quả Holland không? Mình sẽ tư vấn chi tiết hơn dựa vào đó, ví dụ như bạn phù hợp với lập trình, an ninh mạng, quản trị hệ thống hay phân tích dữ liệu.

Bạn chia sẻ thêm chút nhé!

No redirect


### Test 2: ON-TOPIC

Input: Điểm chuẩn ĐH Bách Khoa năm 2025?


Google Search API not configured



Mình chưa truy cập được dữ liệu điểm chuẩn năm 2025 của ĐH Bách Khoa ngay lúc này. Bạn có thể thử kiểm tra trên trang chính thức của trường hoặc các trang tin tức giáo dục uy tín như vnexpress, thanhnien, hoặc cafef nhé.

Nếu bạn muốn, mình có thể giúp bạn tìm hiểu thêm về các ngành hot, xu hướng nghề phù hợp với bạn để chuẩn bị hồ sơ cho ĐH Bách Khoa. Bạn có muốn không?

No redirect


## 5. Test Edge Cases

In [11]:
# Test edge cases (có từ khóa off-topic nhưng context hướng nghiệp)
for i, message in enumerate(test_cases_edge, 1):
    print(f"\n\n### Test {i}: EDGE CASE")
    test_message(message)



### Test 1: EDGE CASE

Input: Tôi muốn học ngành âm nhạc

Ngành âm nhạc là lựa chọn tuyệt vời nếu bạn có đam mê sáng tạo, biểu diễn hoặc nghiên cứu âm thanh. Để tư vấn chuẩn hơn, bạn cho mình biết kết quả Holland (RIASEC) của bạn được không? Nhất là điểm nhóm A (Artistic) của bạn thế nào? 

Nếu bạn có điểm Artistic cao, ngành âm nhạc rất hợp với bạn rồi. Mình sẽ gợi ý nghề, lộ trình học và kỹ năng cần thiết cho bạn. Nếu điểm Artistic chưa cao, mình cũng có thể giúp bạn cân nhắc thêm nhé.

Bạn có thể chia sẻ kết quả Holland của bạn không?

No redirect


### Test 2: EDGE CASE

Input: Ngành phim ảnh học gì?

Ngành Phim ảnh (hay còn gọi là Điện ảnh, Truyền hình) học những gì thì mình tóm gọn cho bạn như sau:

1. **Kiến thức cơ bản về điện ảnh**: Lịch sử điện ảnh, lý thuyết phim, các thể loại phim, ngôn ngữ hình ảnh, cách kể chuyện bằng hình ảnh.

2. **Kỹ năng thực hành**: Quay phim, dựng phim, biên tập, ánh sáng, âm thanh, kỹ thuật quay phim, sử dụng thiết bị máy móc chuyên dụng.

3. **S

## 6. Test với Profile và Holland Data

In [12]:
# Load data từ file
profile_data = load_profile_from_file("User_01.json")
holland_data = load_holland_from_file("Holland_test_u1.json")

if profile_data:
    print(f"Profile: {profile_data.basic_info.full_name}")
if holland_data:
    print(f"Holland: {holland_data.top_code} ({', '.join(holland_data.top3)})")

Profile: Nguyễn Minh Anh
Holland: ASI (A, S, I)


In [13]:
# Test off-topic với profile/holland (guardrail vẫn nên hoạt động)
thread_id = str(uuid.uuid4())

print("\n### Test OFF-TOPIC với Profile/Holland:")
test_message("Hôm nay trời đẹp quá", thread_id, profile_data, holland_data)

print("\n### Test ON-TOPIC với Profile/Holland:")
test_message("Tư vấn nghề cho tôi dựa trên kết quả Holland", thread_id, profile_data, holland_data)


### Test OFF-TOPIC với Profile/Holland:

Input: Hôm nay trời đẹp quá

Case này hơi lệch scope hướng nghiệp của mình rồi  
Mình đang tập trung giúp bạn chọn ngành/chọn nghề cho đúng.  
Bạn muốn đi theo hướng nào nè:  
1) Bạn đang thích ngành gì và đang phân vân điều gì?  
2) Bạn mạnh môn nào / thích hoạt động nào nhất?  
3) Bạn có kết quả Holland (RIASEC) chưa để mình gợi ý nghề?

GUARDRAIL REDIRECT

### Test ON-TOPIC với Profile/Holland:

Input: Tư vấn nghề cho tôi dựa trên kết quả Holland

Ok, bạn có kết quả Holland là ASI, tức là:

- A (Artistic): Bạn có thiên hướng sáng tạo, thích làm việc độc lập, thể hiện cá tính qua nghệ thuật, viết lách, thiết kế...
- S (Social): Bạn thích giúp đỡ người khác, giao tiếp, tư vấn, dạy dỗ, nhạy cảm với cảm xúc người khác.
- I (Investigative): Bạn thích tìm hiểu, phân tích, nghiên cứu, giải quyết vấn đề một cách logic.

Với bộ ba ASI này, bạn rất phù hợp với những nghề vừa sáng tạo, vừa có yếu tố con người, lại có chiều sâu phân tích. Ví dụ:

- Tâm 

## 7. Summary

**Kết quả mong đợi:**
- OFF-TOPIC cases → Có guardrail redirect (template)
- ON-TOPIC cases → Không có guardrail redirect, trả lời bình thường
- EDGE cases → Không có guardrail redirect (vì context hướng nghiệp)

**Template redirect phải có:**
```
Case này hơi lệch scope hướng nghiệp của mình rồi   
Mình đang tập trung giúp bạn chọn ngành/chọn nghề cho đúng.  
Bạn muốn đi theo hướng nào nè:
1) Bạn đang thích ngành gì và đang phân vân điều gì?
2) Bạn mạnh môn nào / thích hoạt động nào nhất?
3) Bạn có kết quả Holland (RIASEC) chưa để mình gợi ý nghề?
```